# MATH240 Python Project: Gauss–Jordan Elimination

## Build a Linear-System Analyzer

In Lecture 1, a linear system is represented by its augmented matrix
\([A\mid b]\). Gauss–Jordan elimination uses elementary row operations to
transform the augmented matrix into reduced row echelon form (RREF). In this
project, you will implement that process and use it to classify and solve
linear systems.

### Learning objectives

After completing the project, you should be able to:

1. implement the three elementary row operations;
2. explain pivoting and produce RREF;
3. compute \(\operatorname{Rank}(A)\) and \(\operatorname{Rank}([A\mid b])\);
4. classify a system as having a unique solution, infinitely many solutions,
   or no solution; and
5. describe the solutions of a homogeneous system in vector form.

### Rules

- Do **not** call a built-in RREF or linear-system solver.
- Use only the three elementary row operations from Lecture 1.
- Keep exact arithmetic by using `Fraction`.
- Use matrices of at least size \(3\times 3\) in your own test case.
- Add brief comments that explain your pivot choices.

### Submission

Submit the completed notebook and a short PDF export showing all outputs.


In [ ]:
from fractions import Fraction
from copy import deepcopy


def F(value):
    """Convert an integer, string, or Fraction to an exact Fraction."""
    return value if isinstance(value, Fraction) else Fraction(value)


def as_fraction_matrix(matrix):
    """Return a deep copy of matrix with every entry stored as a Fraction."""
    if not matrix or not matrix[0]:
        raise ValueError("The matrix must be non-empty.")
    width = len(matrix[0])
    if any(len(row) != width for row in matrix):
        raise ValueError("All rows must have the same number of entries.")
    return [[F(entry) for entry in row] for row in matrix]


def show_matrix(matrix):
    """Print a matrix in a compact, readable form."""
    for row in matrix:
        print("[ " + "  ".join(f"{str(x):>7}" for x in row) + " ]")
    print()


## Part 1 — Elementary row operations (8 marks)

Complete the following functions. Each function must change a copy of the
matrix and return the new matrix.

- `swap_rows(M, i, j)` performs \(R_i\leftrightarrow R_j\).
- `scale_row(M, i, k)` performs \(R_i\times k\), where \(k\ne0\).
- `replace_row(M, i, j, k)` performs \(R_i+kR_j\), where \(i\ne j\).

Python row indices begin at 0, although the printed operation record should
use the mathematical row numbers 1, 2, 3, … .


In [ ]:
def swap_rows(M, i, j):
    A = deepcopy(M)
    A[i], A[j] = A[j], A[i]
    return A


def scale_row(M, i, k):
    k = F(k)
    if k == 0:
        raise ValueError("k can not be zero")
    A = deepcopy(M)
    A[i] = [k * x for x in A[i]]
    return A

def replace_row(M, i, j, k):
    if i == j:
        raise ValueError("i and j must be different")
    k = F(k)
    A = deepcopy(M)
    A[i] = [
        A[i][col] + k * A[j][col]
        for col in range(len(A[i]))
    ]
    return A


In [ ]:
# Quick checks for Part 1
M0 = as_fraction_matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
assert swap_rows(M0, 0, 2) == as_fraction_matrix([[7, 8, 9], [4, 5, 6], [1, 2, 3]])
assert scale_row(M0, 1, Fraction(1, 2))[1] == [2, Fraction(5, 2), 3]
assert replace_row(M0, 2, 0, -7)[2] == [0, -6, -12]
print("Part 1 checks passed.")


## Part 2 — Gauss–Jordan elimination (14 marks)

Complete `gauss_jordan`. Work from left to right through the coefficient
columns. For each pivot column:

1. find a nonzero entry at or below the current pivot row;
2. interchange rows when necessary;
3. multiply the pivot row so that the leading entry becomes 1; and
4. use row replacement to make every other entry in the pivot column zero.

Return both the RREF and a list of the row operations used. Do not pivot in
the final (constant) column of an augmented matrix.


In [ ]:
def gauss_jordan(augmented, show_steps=False):
    M = as_fraction_matrix(augmented)
    operations = []
    pivot_row = 0
    coefficient_columns = len(M[0]) - 1

    for col in range(coefficient_columns):
        nonzero_row = None #finds a non zero entry
        for r in range(pivot_row, len(M)):
            if M[r][col] != 0:
                nonzero_row = r
                break

        # here if the  column below the pivot is zero it moves to next column
        if nonzero_row is None:
            continue

        # here it will swap the rows if it is needed
        if nonzero_row != pivot_row:
            M = swap_rows(M, pivot_row, nonzero_row)

            operations.append(
                f"R{pivot_row + 1} <-> R{nonzero_row + 1}"
            )

            if show_steps:
                print(operations[-1])
                show_matrix(M)

        # this will scale the pivot row so that the pivot becomes 1
        pivot = M[pivot_row][col]
        if pivot != 1:
            factor = Fraction(1, 1) / pivot
            M = scale_row(M, pivot_row, factor)

            operations.append(
                f"R{pivot_row + 1} -> ({factor})R{pivot_row + 1}"
            )

            if show_steps:
                print(operations[-1])
                show_matrix(M)

        # this s makes all other entries in this pivot column zeros
        for r in range(len(M)):
            if r != pivot_row and M[r][col] != 0:
                factor = -M[r][col]
                M = replace_row(M, r, pivot_row, factor)

                operations.append(
                    f"R{r + 1} -> R{r + 1} + ({factor})R{pivot_row + 1}"
                )
                if show_steps:
                    print(operations[-1])
                    show_matrix(M)
        # this moves down to the next pivot row
        pivot_row += 1

        # this will stop the function if there is no more rows
        if pivot_row == len(M):
            break

    return M, operations

In [ ]:
# Lecture-style check: this system has the solution (-2, 2).
lecture_system = [[1, 2, 2], [3, 5, 4]]
lecture_rref, lecture_operations = gauss_jordan(lecture_system)
show_matrix(lecture_rref)
assert lecture_rref == as_fraction_matrix([[1, 0, -2], [0, 1, 2]])
print("Part 2 check passed.")


## Part 3 — Rank and classification (10 marks)

Complete the functions below.

- The rank of a matrix is the number of nonzero rows in an echelon form.
- A system is consistent exactly when
  \(\operatorname{Rank}(A)=\operatorname{Rank}([A\mid b])\).
- A consistent system has a unique solution when the common rank equals the
  number of variables; otherwise it has infinitely many solutions.


In [ ]:
def nonzero_row_count(matrix):
    count = 0
    for row in matrix:
        if any(entry != 0 for entry in row):
            count += 1
    return count

def classify_system(rref_augmented):
    M = as_fraction_matrix(rref_augmented)
    number_of_variables = len(M[0]) - 1
    coefficient_matrix = [row[:-1] for row in M]

    rank_A = nonzero_row_count(coefficient_matrix)
    rank_augmented = nonzero_row_count(M)

    if rank_A != rank_augmented:
        classification = "no solution"
    elif rank_A == number_of_variables:
        classification = "unique solution"
    else:
        classification = "infinitely many solutions"
    return {
        "rank_A": rank_A,
        "rank_augmented": rank_augmented,
        "number_of_variables": number_of_variables,
        "classification": classification
    }

## Part 4 — Analyze three systems (12 marks)

For each augmented matrix below:

1. display the original matrix;
2. display the row operations and RREF;
3. report \(\operatorname{Rank}(A)\) and
   \(\operatorname{Rank}([A\mid b])\); and
4. classify the system.

For the unique case, state the solution. For the infinite case, state the
number of parameters and give the solutions in vector form. For the
inconsistent case, identify the row that represents an impossible equation.


In [ ]:
systems = {
    "System A": [
        [1, 2, -1, 3],
        [2, -1, 1, 2],
        [3, 1, 1, 7],
    ],
    "System B": [
        [1, 2, -1, 1, 3],
        [2, 4, -2, 2, 6],
        [0, 1, 1, -1, 1],
    ],
    "System C": [
        [1, -1, 2, 1],
        [2, -2, 4, 3],
        [0, 1, -1, 0],
    ],
}


In [ ]:
# TODO: analyze System A, System B, and System C.
for name, augmented in systems.items():
    print(name)
    print("Original augmented matrix:")
    show_matrix(as_fraction_matrix(augmented))

    rref, operations = gauss_jordan(augmented)

    print("Row operations:")
    for operation in operations:
        print(operation)

    print("\nRREF:")
    show_matrix(rref)

    info = classify_system(rref)

    print("Rank(A):", info["rank_A"])
    print("Rank([A|b]):", info["rank_augmented"])
    print("Classification:", info["classification"])

    if info["classification"] == "unique solution":
        solution = [row[-1] for row in rref]
        print("Solution:", solution)

    elif info["classification"] == "infinitely many solutions":
        parameters = info["number_of_variables"] - info["rank_A"]
        print("Number of parameters:", parameters)

    elif info["classification"] == "no solution":
        for row_number, row in enumerate(rref, start=1):
            if all(x == 0 for x in row[:-1]) and row[-1] != 0:
                print("Impossible equation is in row", row_number, ":", row)

    print("-" * 40)


## Part 5 — Homogeneous system (6 marks)

Apply your program to the homogeneous system whose matrix of coefficients is

\[
A=\begin{bmatrix}
1&2&-1&3\\
2&4&-2&6\\
0&1&1&-1
\end{bmatrix}.
\]

Add a zero constant column, find the RREF, and answer:

1. What is \(\operatorname{Rank}(A)\)?
2. How many parameters occur?
3. Does the system have nonzero solutions?
4. Give all solutions in vector form.


In [ ]:
A_homogeneous = [
    [1, 2, -1, 3],
    [2, 4, -2, 6],
    [0, 1, 1, -1],
]

# TODO: append the zero column, analyze the system, and give the answer.
# Add the zero constant column
augmented_homogeneous = [row + [0] for row in A_homogeneous]
print("original augmented matrix :")
show_matrix(as_fraction_matrix(augmented_homogeneous))
# here it finds reduced row echlon form
rref, operations = gauss_jordan(augmented_homogeneous)
print("Row operations:")
for operation in operations:
    print(operation)
print("\nRREF:")
show_matrix(rref)
# this will classify the linear system
info = classify_system(rref)
rank_A = info["rank_A"]
number_of_variables = info["number_of_variables"]
parameters = number_of_variables - rank_A

print("Rank(A):", rank_A)
print("Number of parameters:", parameters)

if parameters > 0:
    print("Nonzero solutions exist: Yes")
else:
    print("Nonzero solutions exist: No")

print("Vector form:")
print("x = s(3, -1, 1, 0) + t(-5, 1, 0, 1)")

## Part 6 — Reflection and your own test (optional extension)

Create one augmented matrix with at least three equations and three variables.
Choose it so that a row interchange is necessary at the first pivot. Before
running your code, predict the classification. Then compare your prediction
with the program’s result.

In 100–150 words, explain:

- why exact fractions are useful in Gauss–Jordan elimination;
- how a row of the form \([0\;0\;\cdots\;0\mid c]\), \(c\ne0\), reveals
  inconsistency; and
- how rank determines the number of parameters in a consistent system.


In [ ]:
mysystem = [
    [0, 1, 1, 2],
    [1, 1, 0, 3],
    [2, -1, 1, 1]
]

print("My test system:")
show_matrix(as_fraction_matrix(mysystem))
rref, operations = gauss_jordan(mysystem)

print("Row operations:")
for operation in operations:
    print(operation)

print("\nRREF:")
show_matrix(rref)

info = classify_system(rref)
print(info)

getting exact fractions is very important in gauss jordan elimination because if you get an entry that is zero point something, python might round it, this will give rounding errors, which makes the solution inaccurate, for example we may get a row that is all zeroes point something = c, if it gets rounded, we will get "no solution" for the linear system as a classification which is not true in case of decimals/fractions.